In [38]:
# Install MLflow
#!pip install mlflow --quiet

import mlflow
import mlflow.sklearn
print(f"✅ MLflow version: {mlflow.__version__}")
mlflow.set_tracking_uri("file:../mlruns")

✅ MLflow version: 3.10.1


In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("MLFLOW TRACKING - DIMENSIONALITY REDUCTION EXPERIMENTS")
print("="*70)

# ✅ MLflow setup
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Dimensionality Reduction")

print(f"\n✅ MLflow experiment: Dimensionality Reduction")
print(f"✅ Tracking URI: {mlflow.get_tracking_uri()}")
print(f"✅ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

MLFLOW TRACKING - DIMENSIONALITY REDUCTION EXPERIMENTS

✅ MLflow experiment: Dimensionality Reduction
✅ Tracking URI: file:../mlruns
✅ Timestamp: 2026-04-28 14:17:30


In [40]:
import mlflow
import os

TRACKING_PATH = "file:///C:/Users/nlalr/Desktop/Rama/PatrollQ_Project/Data_prep/mlruns"

mlflow.set_experiment("Debug_Test")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Working dir:", os.getcwd())

Tracking URI: file:../mlruns
Working dir: c:\Users\nlalr\Desktop\Rama\PatrollQ_Project\Data_prep


In [41]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:../mlruns


In [42]:
with mlflow.start_run():
    mlflow.log_param("debug_param", 1)
    mlflow.log_metric("debug_metric", 0.99)
    print("Run ID:", mlflow.active_run().info.run_id)

Run ID: 4b44389f6a004ce79c706de97626a877


In [43]:
# Load original data
df = pd.read_csv('../Data/chicago_crime_with_features.csv')
X_scaled = pd.read_csv('../Data/feature_matrix_scaled.csv')

# Load your PCA results
df_pca = pd.read_csv('../Data/pca_components.csv')
feature_importance = pd.read_csv('../Data/pca_feature_importance.csv')

print(f"\n📊 Data loaded:")
print(f"   Original records: {len(df):,}")
print(f"   Original features: {X_scaled.shape[1]}")
print(f"   PCA components: {df_pca.shape[1]}")


📊 Data loaded:
   Original records: 496,290
   Original features: 28
   PCA components: 2


In [44]:
def create_aggregated_features(X_scaled):
    """
    Create aggregated features using domain knowledge
    (Same as your Cell 2 from PCA notebook)
    """
    # Get base features
    all_cols = X_scaled.columns.tolist()
    
    encoded_features = [col for col in all_cols if '_Type_' in col or 
                       '_Description_' in col or '_Location_' in col or
                       '_Block_' in col or '_Beat_' in col or
                       '_FBI_' in col or '_IUCR_' in col]
    
    base_features = [col for col in all_cols if col not in encoded_features]
    X_base = X_scaled[base_features].copy()
    
    aggregated_features = {}
    
    # Spatial features
    if 'Latitude_Scaled' in X_base.columns and 'Longitude_Scaled' in X_base.columns:
        aggregated_features['Spatial_NorthSouth'] = X_base['Latitude_Scaled']
        aggregated_features['Spatial_EastWest'] = X_base['Longitude_Scaled']
        
        if 'Distance_From_Center' in X_base.columns:
            aggregated_features['Spatial_Centrality'] = X_base['Distance_From_Center']
    
    # Administrative zones
    admin_features = []
    if 'District' in X_base.columns:
        admin_features.append(X_base['District'])
    if 'Ward' in X_base.columns:
        admin_features.append(X_base['Ward'])
    if 'Community Area' in X_base.columns:
        admin_features.append(X_base['Community Area'])
    
    if len(admin_features) > 0:
        aggregated_features['Administrative_Zone'] = pd.concat(admin_features, axis=1).mean(axis=1)
    
    # Temporal features
    if 'Hour' in X_base.columns:
        aggregated_features['Time_OfDay'] = X_base['Hour']
    
    day_pattern_score = pd.Series(0, index=X_base.index)
    if 'DayOfWeek' in X_base.columns:
        day_pattern_score += X_base['DayOfWeek'] * 0.5
    if 'IsWeekend' in X_base.columns:
        day_pattern_score += X_base['IsWeekend'] * 3
    if 'Day' in X_base.columns:
        day_pattern_score += (X_base['Day'] / 31) * 0.3
    aggregated_features['Day_Pattern'] = day_pattern_score
    
    activity_score = pd.Series(0, index=X_base.index)
    if 'IsRushHour' in X_base.columns:
        activity_score += X_base['IsRushHour'] * 2
    if 'IsLateNight' in X_base.columns:
        activity_score += X_base['IsLateNight'] * 1.5
    if 'IsWeekend' in X_base.columns:
        activity_score += X_base['IsWeekend'] * 1
    aggregated_features['Activity_Level'] = activity_score
    
    if 'Month' in X_base.columns:
        aggregated_features['Season'] = X_base['Month']
    
    # Crime pattern features
    crime_intensity = pd.Series(0, index=X_base.index)
    if 'Severity' in X_base.columns:
        crime_intensity += X_base['Severity'] * 0.4
    if 'Location_Crime_Frequency' in X_base.columns:
        crime_intensity += X_base['Location_Crime_Frequency'] * 0.4
    if 'IsHighCrimeDistrict' in X_base.columns:
        crime_intensity += X_base['IsHighCrimeDistrict'] * 0.2
    aggregated_features['Crime_Intensity'] = crime_intensity
    
    # Create DataFrame
    X_agg = pd.DataFrame(aggregated_features, index=X_base.index)
    
    # Remove low variance features
    feature_vars = X_agg.var()
    X_agg = X_agg.loc[:, feature_vars > 0.01]
    
    return X_agg

print("✅ Feature engineering function defined")

✅ Feature engineering function defined


In [45]:
import mlflow
import os
import tempfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ✅ Always set tracking + ONE experiment
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Dimensionality Reduction")

print("\n" + "="*70)
print("EXPERIMENT: PCA (2 Components, Aggregated Features)")
print("="*70)

try:
    with mlflow.start_run(run_name="PCA_2_components_aggregated"):
        
        print("🚀 Run started")

        # 🔹 Feature Engineering
        X_aggregated = create_aggregated_features(X_scaled)
        print("Shape after aggregation:", X_aggregated.shape)

        # 🔹 PCA
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_aggregated)

        # 🔹 Metrics
        variance_explained = pca.explained_variance_ratio_.sum()
        pc1_variance = pca.explained_variance_ratio_[0]
        pc2_variance = pca.explained_variance_ratio_[1]

        # 🔹 Log params
        mlflow.log_param("n_components", 2)
        mlflow.log_param("feature_engineering", "aggregated")
        mlflow.log_param("n_input_features", X_aggregated.shape[1])
        mlflow.log_param("approach", "domain_knowledge_aggregation")

        # 🔹 Log metrics
        mlflow.log_metric("total_variance_explained", variance_explained)
        mlflow.log_metric("pc1_variance", pc1_variance)
        mlflow.log_metric("pc2_variance", pc2_variance)
        mlflow.log_metric("reduction_ratio", X_aggregated.shape[1] / 2)
        mlflow.log_metric("meets_70_percent_target", int(variance_explained >= 0.70))

        # 🔹 Log model
        mlflow.sklearn.log_model(pca, "pca_model")

        # 🔹 Plot
        fig, ax = plt.subplots(figsize=(10, 6))
        components = ['PC1', 'PC2']
        variances = pca.explained_variance_ratio_

        ax.bar(components, variances)
        ax.axhline(y=0.70, linestyle='--', linewidth=2)
        ax.set_ylabel('Variance Explained')
        ax.set_title('PCA Variance Explained')
        ax.set_ylim(0, 0.7)

        plt.tight_layout()

        mlflow.log_figure(fig, "variance_explained.png")
        plt.close()

        # 🔹 Feature importance
        feature_importance_df = pd.DataFrame({
            'Feature': X_aggregated.columns,
            'Importance': np.sum(np.abs(pca.components_), axis=0)
        }).sort_values('Importance', ascending=False)

        # ✅ Use temp directory (clean practice)
        with tempfile.TemporaryDirectory() as tmpdir:

            fi_path = os.path.join(tmpdir, "feature_importance.csv")
            feature_importance_df.to_csv(fi_path, index=False)
            mlflow.log_artifact(fi_path)

            summary = f"""
PCA Optimal Solution Summary
----------------------------
Total Variance: {variance_explained*100:.2f}%
PC1: {pc1_variance*100:.2f}%
PC2: {pc2_variance*100:.2f}

Input Features: {X_aggregated.shape[1]}

Top Features:
1. {feature_importance_df.iloc[0]['Feature']}
2. {feature_importance_df.iloc[1]['Feature']}
3. {feature_importance_df.iloc[2]['Feature']}
"""

            summary_path = os.path.join(tmpdir, "summary.txt")
            with open(summary_path, "w", encoding="utf-8") as f:
                f.write(summary)

            mlflow.log_artifact(summary_path)

        print("\n✅ Experiment logged successfully!")
        print(f"Variance: {variance_explained*100:.2f}%")
        print(f"Run ID: {mlflow.active_run().info.run_id}")

except Exception as e:
    print("❌ ERROR:", e)


EXPERIMENT: PCA (2 Components, Aggregated Features)
🚀 Run started


2026/04/28 14:17:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Shape after aggregation: (496290, 9)


2026/04/28 14:17:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 14:17:41 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



✅ Experiment logged successfully!
Variance: 75.01%
Run ID: e34b63b507004117a8e27e7ee16d4ff7


In [46]:
import mlflow
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Dimensionality Reduction")

print("\n" + "="*70)
print("EXPERIMENT: PCA with 3 Components")
print("="*70)

with mlflow.start_run(run_name="PCA_3_components_aggregated"):

    print("🚀 Run started")

    X_aggregated = create_aggregated_features(X_scaled)
    print("Shape:", X_aggregated.shape)

    # PCA
    pca_3 = PCA(n_components=3)
    X_pca_3 = pca_3.fit_transform(X_aggregated)

    # Metrics
    variance_explained = pca_3.explained_variance_ratio_.sum()

    # Params
    mlflow.log_param("n_components", 3)
    mlflow.log_param("feature_engineering", "aggregated")
    mlflow.log_param("n_input_features", X_aggregated.shape[1])
    mlflow.log_param("approach", "domain_knowledge_aggregation")

    # Metrics
    mlflow.log_metric("total_variance_explained", variance_explained)
    mlflow.log_metric("pc1_variance", pca_3.explained_variance_ratio_[0])
    mlflow.log_metric("pc2_variance", pca_3.explained_variance_ratio_[1])
    mlflow.log_metric("pc3_variance", pca_3.explained_variance_ratio_[2])
    mlflow.log_metric("reduction_ratio", X_aggregated.shape[1] / 3)
    mlflow.log_metric("meets_70_percent_target", int(variance_explained >= 0.70))

    # Model
    mlflow.sklearn.log_model(pca_3, "pca_model")

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    variances = pca_3.explained_variance_ratio_

    ax1.bar(['PC1', 'PC2', 'PC3'], variances)
    ax1.set_title('Individual Variance')

    cumulative = np.cumsum(variances)
    ax2.plot(range(1, 4), cumulative, marker='o')
    ax2.axhline(y=0.70, linestyle='--')
    ax2.set_title('Cumulative Variance')

    plt.tight_layout()
    mlflow.log_figure(fig, "variance_comparison.png")
    plt.close()

    print("\n✅ Experiment logged successfully!")
    print(f"Variance: {variance_explained*100:.2f}%")
    print(f"Run ID: {mlflow.active_run().info.run_id}")


EXPERIMENT: PCA with 3 Components
🚀 Run started


2026/04/28 14:17:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 14:17:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Shape: (496290, 9)


2026/04/28 14:17:51 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



✅ Experiment logged successfully!
Variance: 82.22%
Run ID: 7bc1401ab4344f569a32a470190ad11b


In [47]:
import mlflow
import numpy as np
from sklearn.decomposition import PCA

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Dimensionality Reduction")

print("\n" + "="*70)
print("EXPERIMENT: Baseline PCA (No Feature Engineering)")
print("="*70)

with mlflow.start_run(run_name="PCA_baseline_no_aggregation"):

    print("🚀 Run started")

    # 🔹 Base features (remove encoded)
    all_cols = X_scaled.columns.tolist()

    encoded_features = [col for col in all_cols if '_Type_' in col or 
                        '_Description_' in col or '_Location_' in col or
                        '_Block_' in col or '_Beat_' in col or
                        '_FBI_' in col or '_IUCR_' in col]

    base_features = [col for col in all_cols if col not in encoded_features]
    X_base = X_scaled[base_features].copy()

    print("Base shape:", X_base.shape)

    # 🔹 Find optimal components
    pca_full = PCA()
    pca_full.fit(X_base)

    cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
    n_components_70 = np.argmax(cumulative_var >= 0.70) + 1

    print("Optimal components for 70%:", n_components_70)

    # 🔹 Final PCA
    pca_baseline = PCA(n_components=n_components_70)
    X_pca_baseline = pca_baseline.fit_transform(X_base)

    variance_explained = pca_baseline.explained_variance_ratio_.sum()

    # 🔹 Params
    mlflow.log_param("n_components", int(n_components_70))
    mlflow.log_param("feature_engineering", "none")
    mlflow.log_param("n_input_features", X_base.shape[1])
    mlflow.log_param("approach", "baseline_no_aggregation")

    # 🔹 Metrics
    mlflow.log_metric("total_variance_explained", variance_explained)

    for i, var in enumerate(pca_baseline.explained_variance_ratio_):
        mlflow.log_metric(f"pc{i+1}_variance", var)

    mlflow.log_metric("reduction_ratio", X_base.shape[1] / n_components_70)
    mlflow.log_metric("meets_70_percent_target", int(variance_explained >= 0.70))

    # 🔹 Model
    mlflow.sklearn.log_model(pca_baseline, "pca_model")

    print("\n✅ Baseline PCA logged successfully!")
    print(f"Components needed: {n_components_70}")
    print(f"Variance: {variance_explained*100:.2f}%")
    print(f"Run ID: {mlflow.active_run().info.run_id}")


EXPERIMENT: Baseline PCA (No Feature Engineering)
🚀 Run started
Base shape: (496290, 27)
Optimal components for 70%: 11


2026/04/28 14:17:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 14:17:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/28 14:17:57 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!



✅ Baseline PCA logged successfully!
Components needed: 11
Variance: 73.73%
Run ID: c6c38cb96ef34202bb9987a840c72d9a


In [48]:
import mlflow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("Dimensionality Reduction")  # ✅ SAME experiment

print("\n" + "="*70)
print("EXPERIMENT: PCA (Fast Sampled Version)")
print("="*70)

with mlflow.start_run(run_name="PCA_fast_sampled"):

    print("🚀 Run started")

    X_aggregated = create_aggregated_features(X_scaled)
    print("Full shape:", X_aggregated.shape)

    # 🔹 Sampling
    SAMPLE_SIZE = 1000
    np.random.seed(42)

    sample_idx = np.random.choice(
        len(X_aggregated),
        size=min(SAMPLE_SIZE, len(X_aggregated)),
        replace=False
    )

    X_sample = X_aggregated.iloc[sample_idx]
    print("Sample shape:", X_sample.shape)

    # 🔹 PCA
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_sample)

    # 🔹 Params
    mlflow.log_param("algorithm", "PCA")
    mlflow.log_param("n_components", 2)
    mlflow.log_param("sample_size", len(X_sample))
    mlflow.log_param("n_input_features", X_aggregated.shape[1])
    mlflow.log_param("approach", "fast_sampled")

    # 🔹 Metrics
    var1 = float(pca.explained_variance_ratio_[0])
    var2 = float(pca.explained_variance_ratio_[1])
    var_sum = float(pca.explained_variance_ratio_.sum())

    mlflow.log_metric("pc1_variance", var1)
    mlflow.log_metric("pc2_variance", var2)
    mlflow.log_metric("total_variance_explained", var_sum)

    # 🔹 Visualization
    df_sample = df.iloc[sample_idx]
    crime_types = df_sample["Primary Type"]
    top_crimes = crime_types.value_counts().head(5).index

    fig, ax = plt.subplots(figsize=(12, 8))

    for crime in top_crimes:
        mask = crime_types == crime
        ax.scatter(
            X_pca[mask, 0],
            X_pca[mask, 1],
            label=crime,
            alpha=0.6,
            s=20
        )

    ax.set_xlabel("PCA Component 1")
    ax.set_ylabel("PCA Component 2")
    ax.set_title("PCA Visualization (Top 5 Crime Types)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    mlflow.log_figure(fig, "pca_visualization.png")
    plt.close()

    print("\n✅ Fast PCA experiment logged successfully!")
    print(f"Variance: {var_sum:.4f}")
    print(f"Run ID: {mlflow.active_run().info.run_id}")


EXPERIMENT: PCA (Fast Sampled Version)
🚀 Run started
Full shape: (496290, 9)
Sample shape: (1000, 9)

✅ Fast PCA experiment logged successfully!
Variance: 0.7567
Run ID: 768f636511484ae29bb6d481dd5e59d1


In [49]:
print("\n" + "="*70)
print("COMPARING ALL EXPERIMENTS")
print("="*70)

# ✅ Get experiment safely
experiment = mlflow.get_experiment_by_name("Dimensionality Reduction")

if experiment is None:
    print("❌ Experiment not found!")
else:
    runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

    print("\nExperiment Comparison:")
    print("="*100)

    # Desired columns
    comparison_cols = [
        'run_id',
        'tags.mlflow.runName',
        'params.n_components',
        'params.feature_engineering',
        'metrics.total_variance_explained',
        'params.n_input_features'
    ]

    # Keep only available columns
    existing_cols = [col for col in comparison_cols if col in runs_df.columns]

    comparison_df = runs_df[existing_cols].copy()

    # ✅ Safe rename (only if present)
    rename_map = {
        'run_id': 'Run ID',
        'tags.mlflow.runName': 'Run Name',
        'params.n_components': 'Components',
        'params.feature_engineering': 'Feature Eng',
        'metrics.total_variance_explained': 'Variance',
        'params.n_input_features': 'Input Features'
    }

    comparison_df.rename(columns=rename_map, inplace=True)

    # ✅ Format variance nicely
    if 'Variance' in comparison_df.columns:
        comparison_df['Variance'] = comparison_df['Variance'].apply(
            lambda x: f"{x*100:.2f}%" if pd.notna(x) else "N/A"
        )

    # ✅ Sort by best performance
    if 'Variance' in comparison_df.columns:
        comparison_df = comparison_df.sort_values(
            by='Variance',
            ascending=False
        )

    print(comparison_df.to_string(index=False))

    print("\n" + "="*100)
    print("✅ All experiments tracked in MLflow!")
    print(f"Total runs: {len(runs_df)}")

    print("\nView MLflow UI:")
    print("   Run in terminal: mlflow ui --port 5050")
    print("   Then open: http://localhost:5050")


COMPARING ALL EXPERIMENTS

Experiment Comparison:
                          Run ID                           Run Name Components Feature Eng Variance Input Features
7bc1401ab4344f569a32a470190ad11b        PCA_3_components_aggregated          3  aggregated   82.22%              9
a04f8834777a422db0a2df0211068ae0        PCA_3_components_aggregated          3  aggregated   82.22%              9
768f636511484ae29bb6d481dd5e59d1                   PCA_fast_sampled          2        None   75.67%              9
1ed5dc57ba464431baaf5a963618b5d2                   PCA_fast_sampled          2        None   75.67%              9
e34b63b507004117a8e27e7ee16d4ff7        PCA_2_components_aggregated          2  aggregated   75.01%              9
57de86a082314dfd8a4b484f0ec8a872        PCA_2_components_aggregated          2  aggregated   75.01%              9
94f93afa110c4393a4083ed0e70bd184 PCA_Optimal_2Components_Aggregated          2  aggregated   75.01%              9
c6c38cb96ef34202bb9987a840c72

In [50]:
import mlflow
from datetime import datetime
import os

print("\n" + "="*70)
print("GENERATING MLFLOW EXPERIMENT REPORT")
print("="*70)

mlflow.set_tracking_uri("file:../mlruns")

# ✅ Get experiment
experiment = mlflow.get_experiment_by_name("Dimensionality Reduction")

if experiment is None:
    print("❌ Experiment not found!")
else:
    runs_df = mlflow.search_runs([experiment.experiment_id])

    if runs_df.empty:
        print("❌ No runs found!")
    else:
        # ✅ Sort by best variance
        runs_df = runs_df.sort_values(
            by="metrics.total_variance_explained",
            ascending=False
        )

        best_run = runs_df.iloc[0]

        best_variance = best_run.get("metrics.total_variance_explained", None)
        best_components = best_run.get("params.n_components", "N/A")
        best_method = best_run.get("params.feature_engineering", "N/A")

        report = f"""
MLflow Experiment Tracking Report
==================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Experiment: Dimensionality Reduction

Best Model:
-----------
- Components: {best_components}
- Feature Engineering: {best_method}
- Variance Explained: {best_variance*100:.2f}% if best_variance else "N/A"

Total Runs Tracked:
-------------------
{len(runs_df)}

Key Insights:
-------------
- Feature aggregation significantly reduces dimensionality
- PCA provides optimal linear reduction
- MLflow ensures reproducibility and tracking

Next Steps:
-----------
1. Use PCA features for classification models
2. Track ML models using MLflow
3. Register best model in MLflow Registry
"""

        print(report)

        # ✅ Safe save
        os.makedirs("../Data", exist_ok=True)

        file_path = "../Data/mlflow_experiment_report.txt"
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(report)

        print(f"\n✅ Report saved to: {file_path}")


GENERATING MLFLOW EXPERIMENT REPORT

MLflow Experiment Tracking Report
Date: 2026-04-28 14:18:04
Experiment: Dimensionality Reduction

Best Model:
-----------
- Components: 3
- Feature Engineering: aggregated
- Variance Explained: 82.22% if best_variance else "N/A"

Total Runs Tracked:
-------------------
9

Key Insights:
-------------
- Feature aggregation significantly reduces dimensionality
- PCA provides optimal linear reduction
- MLflow ensures reproducibility and tracking

Next Steps:
-----------
1. Use PCA features for classification models
2. Track ML models using MLflow
3. Register best model in MLflow Registry


✅ Report saved to: ../Data/mlflow_experiment_report.txt


print("\n" + "="*70)
print("MLFLOW UI INSTRUCTIONS")
print("="*70)

instructions = """
🚀 How to View Your Experiments in MLflow UI:

1. Open a NEW terminal (not this notebook)

2. Navigate to your project directory:
   cd /path/to/your/project

3. Start MLflow UI:
   mlflow ui

4. Open browser and go to:
   http://localhost:5000

5. In the UI you can:
   ✅ Compare all experiment runs side-by-side
   ✅ View visualizations (variance plots, t-SNE)
   ✅ Download models and artifacts
   ✅ See parameter and metric history
   ✅ Filter and sort experiments

6. To stop MLflow UI:
   Press Ctrl+C in the terminal

📊 What You'll See:
-------------------
- Experiment: Chicago_Crime_Dimensionality_Reduction
- 4 runs with different approaches
- Metrics, parameters, and artifacts for each run
- Visual comparison charts
"""

print(instructions)

In [51]:
print(runs_df.columns)

Index(['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time',
       'end_time', 'metrics.pc1_variance', 'metrics.pc2_variance',
       'metrics.total_variance_explained', 'metrics.pc8_variance',
       'metrics.pc9_variance', 'metrics.pc5_variance',
       'metrics.meets_70_percent_target', 'metrics.pc3_variance',
       'metrics.pc11_variance', 'metrics.pc10_variance',
       'metrics.pc6_variance', 'metrics.pc4_variance',
       'metrics.reduction_ratio', 'metrics.pc7_variance', 'params.sample_size',
       'params.n_input_features', 'params.approach', 'params.algorithm',
       'params.n_components', 'params.feature_engineering',
       'tags.mlflow.runName', 'tags.mlflow.source.type',
       'tags.mlflow.source.name', 'tags.mlflow.user'],
      dtype='object')


# Register Best PCA Model in MLflow Model Registry
This cell demonstrates how to register the best PCA model (from Experiment 1) to the MLflow Model Registry and transition it to the 'Staging' stage.

In [52]:
import mlflow
from mlflow.tracking import MlflowClient

print("\n" + "="*70)
print("REGISTERING BEST PCA MODEL")
print("="*70)

# ✅ Always set tracking
mlflow.set_tracking_uri("file:../mlruns")

# ✅ Correct experiment name
experiment = mlflow.get_experiment_by_name("Dimensionality Reduction")

if experiment is None:
    print("❌ Experiment not found!")
else:
    runs_df = mlflow.search_runs([experiment.experiment_id])

    if runs_df.empty:
        print("❌ No runs found!")
    elif "metrics.total_variance_explained" not in runs_df.columns:
        print("❌ Required metric not found in runs!")
        print("Available columns:", runs_df.columns)
    else:
        # ✅ Sort and pick best run
        runs_df = runs_df.sort_values(
            by="metrics.total_variance_explained",
            ascending=False
        )

        best_run = runs_df.iloc[0]
        best_run_id = best_run["run_id"]

        print(f"\n🏆 Best Run ID: {best_run_id}")
        print(f"Variance: {best_run['metrics.total_variance_explained']:.4f}")

        # ✅ Register model
        model_uri = f"runs:/{best_run_id}/pca_model"
        model_name = "ChicagoCrimePCA"

        result = mlflow.register_model(model_uri, model_name)

        # ✅ Move to Staging
        client = MlflowClient()
        client.transition_model_version_stage(
            name=model_name,
            version=result.version,
            stage="Staging"
        )

        print("\n✅ Model Registered Successfully!")
        print(f"Model Name: {model_name}")
        print(f"Version: {result.version}")
        print("Stage: Staging")

        print("\n👉 Open MLflow UI → Models tab to view")


REGISTERING BEST PCA MODEL

🏆 Best Run ID: 7bc1401ab4344f569a32a470190ad11b

Registered model 'ChicagoCrimePCA' already exists. Creating a new version of this model...
2026/04/28 14:18:05 WARNING mlflow.tracking._model_registry.fluent: Run with id 7bc1401ab4344f569a32a470190ad11b has no artifacts at artifact path 'pca_model', registering model based on models:/m-77be2d1da2924e3ab3671fdeb22f074e instead



Variance: 0.8222


Created version '2' of model 'ChicagoCrimePCA'.



✅ Model Registered Successfully!
Model Name: ChicagoCrimePCA
Version: 2
Stage: Staging

👉 Open MLflow UI → Models tab to view
